In [ ]:
from time import sleep as delay
from project.el5600 import project
if "ic" in globals():
    ic.close()
ic = project(device="el5600", revision="a1", emulator="cp2112", logging=False, is_gui=False)

from phy.multimeter import siglent_sdm3055_auto
from phy.power_supply import keithley_2460, rigol_dp821a, keysight_e36232a
from phy.scope import tektronix_dpo4104b_usb
from phy.eloader import sdl1020x
from phy.relay_16ch import relay_box
# from phy.chamber_th3 import chamber

%matplotlib tk
from interface.docs.output_chart import plot
from interface.cui_colors import color
from interface.i2c_bridge.tca9548a import tca9548
from interface.docs.output_excel import excel_frame, style
from interface.cui_logger import logger as log

import numpy as np

chart = plot()

dm = siglent_sdm3055_auto()
ps = rigol_dp821a()
ks = keysight_e36232a()
kt = keithley_2460()
ld = sdl1020x()
sc = tektronix_dpo4104b_usb()

# dma = agilent_34401a("COM5")
relay = relay_box(i2c_h=ic)
# tc = chamber(port=3)
# mux = tca9548(ic.i2c_h, 0x70)


# ==================================
def disable_all_ps(kt=kt, ps=ps):
    kt.disable
    ks.disable
    ld.disable
    ps.ch1.disable
    ps.ch2.disable
# ==================================

disable_all_ps()

# Test A10
- kt : VDD3V3
- ps.ch1 : relay
- ks : ---> dm.ch1 (alternate IOUT) ---> relay.ch1
- ps.ch2 : ---> relay.ch3 (EN)
- dm.ch1 : ---> relay.ch1 (VIN)
- dm.ch3 : ---> VIN-VOUT diff
- relay.ch1 : VIN
- relay.ch2 : terminal_1=VOUT, terminal_2=loader
- relay.ch3 : EN
- loader : <--- relay.ch2 (100mA load)

In [ ]:
temperature = "N40C"

log.output_set_filename(f"[{temperature}_10] RDS_ON")
log.output_csv(["VIN (V)", "VDD3V3 (V)", "EN (V)", "VIN-VOUT (V)", "I_VOUT (mA)", "DIODE_MODE_EN", "SW_STS", "RRG_STS", "RDS_ON (mo-hm)"])

In [ ]:
disable_all_ps()
relay.init_channels

v_vdd = 3.3
v_en = 3.3

v_start  = 5
v_finish = 30.1
step = 1
ndigit = 1

list_temp = list(np.arange(v_start, v_finish, step))
list_vin = [round(num, ndigit) for num in list_temp]

print(f"voltage step : {step}, round : {ndigit}")

# --------------------------------------------
ps.ch1.cfg_all = 5, 1 # relay
ps.ch1.enable

# --------------------------------------------
kt.cfg_all = v_vdd, 0.01 # vdd
kt.enable
delay(0.5)

# --------------------------------------------
ks.cfg_all = list_vin[0], 0.5 # vin
ks.enable
delay(0.5)

dm.ch1.current_200mA
relay.ch1.enable # monitoring for vin
relay.ch2.enable # vout
delay(0.5)

# --------------------------------------------
ps.ch2.cfg_all = 3.3, 0.1
ps.ch2.enable
delay(0.5)

relay.ch3.enable # en
delay(0.5)

# --------------------------------------------
ic.DIODE_MODE_EN = 0
ld.iset = 0.101
ld.enable

In [ ]:
# ----------------------------------------------------------------------------

for vin in list_vin:
    
    ks.vset = vin
    
    vin_vout = round(dm.ch3.voltage_200mV * 1e+3, 6) # mV scale
    iout = round(dm.ch1.current_200mA, 6)
    reg_diode = ic.DIODE_MODE_EN
    sw_sts = ic.SW_STS
    rrg_sts = ic.RRG_STS
    
    rds_on = round(vin_vout/iout, 6)
    
    ret = [vin, v_vdd, v_en, vin_vout, iout, reg_diode, sw_sts, rrg_sts, rds_on]
    log.output_csv(ret)
    
    print(ret)

# ----------------------------------------------------------------------------

relay.init_channels
disable_all_ps()